# Assignment 1 — Answers

Solutions to all four questions, with explanations and runnable demonstrations.

## Question 1 — Rate Limiter Bugs [15 marks]

**Original buggy code:**

```python
import time

def rate_limit(max_calls: int, period: int):
    def decorator(func):
        calls = []
        def wrapper(*args, **kwargs):
            now = time.time()
            # Remove timestamps older than the period window
            calls = [t for t in calls if now - t < period]
            if len(calls) >= max_calls:
                raise Exception("Rate limit exceeded")
            calls.append(now)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@rate_limit(max_calls=3, period=10)
def fetch_user_data(user_id):
    return f"Data for {user_id}"
```

### i. Bug 1 — `UnboundLocalError`

Inside `wrapper`, the line `calls = [t for t in calls if ...]` **assigns** to `calls`. Because a name is assigned anywhere inside a function, Python treats it as local to that function for the *entire* body — even on the line where it's first read. So the `calls` on the right-hand side of the list comprehension is already treated as the *local* `calls`, which hasn't been assigned yet at that point in execution. This raises `UnboundLocalError: local variable 'calls' referenced before assignment`.

### ii. Bug 2 — Shared state across instances

`calls` lives in the closure of `decorator`, created once when the decorator is applied. That means there is exactly **one shared `calls` list for the whole decorated function**, not one per object. If `fetch_user_data` were a method called on two different instances, both instances would share and compete for the same rate-limit budget. The fix is to track call history **per instance**, keyed on `self` (e.g. via `id(self)`), instead of a single flat list.

### iii. Refactor

Bug 1 alone is fixed by mutating the *same* `calls` list in place (`calls[:] = ...`) instead of reassigning it — that keeps `calls` bound to the enclosing scope's variable, so there's no local-before-assignment problem. That's enough to fix the exact `fetch_user_data` example below, which is a plain function.

Bug 2 only shows up once the decorator is applied to an **instance method** (the question's premise). For that case we additionally key the call history on `self` (`args[0]`), so each instance tracks its own calls independently. Both fixes are shown below.


In [7]:
import time

def rate_limit(max_calls: int, period: int):
    """Fixes Bug 1: mutate `calls` in place instead of reassigning it,
    so it stays bound to the enclosing (decorator) scope."""
    def decorator(func):
        calls = []
        def wrapper(*args, **kwargs):
            now = time.time()
            calls[:] = [t for t in calls if now - t < period]
            if len(calls) >= max_calls:
                raise Exception("Rate limit exceeded")
            calls.append(now)
            return func(*args, **kwargs)
        return wrapper
    return decorator


@rate_limit(max_calls=3, period=10)
def fetch_user_data(user_id):
    return f"Data for {user_id}"


**Demo — Bug 1 fixed: rate limit trips after 3 calls within the period:**

In [10]:
for i in range(4):
    try:
        print(fetch_user_data(i))
    except Exception as e:
        print("Blocked:", e)


Data for 0
Data for 1
Data for 2
Blocked: Rate limit exceeded


**Bug 2 fixed — per-instance keying, for use as a method decorator:**

In [13]:
def rate_limit_per_instance(max_calls: int, period: int):
    """Fixes Bug 2: keys the call history on `self` (the instance),
    so each instance of a class gets its own independent rate limit."""
    def decorator(func):
        calls_by_instance = {}   # id(self) -> [timestamps]

        def wrapper(self, *args, **kwargs):
            key = id(self)
            now = time.time()
            calls = calls_by_instance.setdefault(key, [])
            calls[:] = [t for t in calls if now - t < period]
            if len(calls) >= max_calls:
                raise Exception("Rate limit exceeded")
            calls.append(now)
            return func(self, *args, **kwargs)
        return wrapper
    return decorator


class Client:
    def __init__(self, name):
        self.name = name

    @rate_limit_per_instance(max_calls=2, period=10)
    def call(self):
        return f"{self.name} made a call"

a = Client("A")
b = Client("B")

print(a.call())
print(a.call())
try:
    print(a.call())   # A's 3rd call -> blocked
except Exception as e:
    print("A blocked:", e)

print(b.call())   # B is unaffected by A's usage
print(b.call())


A made a call
A made a call
A blocked: Rate limit exceeded
B made a call
B made a call


## Question 2 — Event Dispatcher (Observer Pattern)

Requirements:
- `subscribe(event_type, callback)` — register a callback for an event type.
- `unsubscribe(event_type, callback)` — remove a registered callback.
- `dispatch(event_type, *args, **kwargs)` — run all callbacks for an event type, in registration order.
- Callbacks must run in the exact order registered.
- If a callback raises, log/print the error but keep running the remaining callbacks.


In [4]:
class EventDispatcher:
    def __init__(self):
        self._listeners: dict[str, list[callable]] = {}

    def subscribe(self, event_type: str, callback: callable):
        self._listeners.setdefault(event_type, []).append(callback)

    def unsubscribe(self, event_type: str, callback: callable):
        if event_type in self._listeners:
            try:
                self._listeners[event_type].remove(callback)
            except ValueError:
                pass  # callback wasn't registered - ignore silently

    def dispatch(self, event_type: str, *args, **kwargs):
        # iterate over a copy so unsubscribing during dispatch is safe
        for callback in list(self._listeners.get(event_type, [])):
            try:
                callback(*args, **kwargs)
            except Exception as e:
                print(f"Error in callback {getattr(callback, '__name__', callback)}: {e}")


**Demo:**

In [5]:
dispatcher = EventDispatcher()

def on_login_1(user):
    print(f"[1] Welcome, {user}!")

def on_login_2(user):
    print(f"[2] Logging login event for {user}")

def on_login_broken(user):
    raise RuntimeError("Something went wrong in this handler")

def on_login_3(user):
    print(f"[3] Sending notification to {user}")

dispatcher.subscribe("login", on_login_1)
dispatcher.subscribe("login", on_login_2)
dispatcher.subscribe("login", on_login_broken)
dispatcher.subscribe("login", on_login_3)

dispatcher.dispatch("login", "alice")

print("---after unsubscribing on_login_2---")
dispatcher.unsubscribe("login", on_login_2)
dispatcher.dispatch("login", "bob")


[1] Welcome, alice!
[2] Logging login event for alice
Error in callback on_login_broken: Something went wrong in this handler
[3] Sending notification to alice
---after unsubscribing on_login_2---
[1] Welcome, bob!
Error in callback on_login_broken: Something went wrong in this handler
[3] Sending notification to bob


## Question 3 — `Typed` Data Descriptor [15 marks]

Requirements:
- `Typed(expected_type)` stores the expected type.
- `__set__` raises `TypeError` with message `f"Expected {expected_type.__name__}, got {type(value).__name__}"` if the value isn't an instance of the expected type.
- `__set_name__` auto-derives the storage attribute name, avoiding manual string keys / collisions.


In [6]:
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        # Called automatically at class-creation time with the attribute name,
        # so each Typed instance gets its own private storage key.
        self.name = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"Expected {self.expected_type.__name__}, got {type(value).__name__}"
            )
        instance.__dict__[self.name] = value


**Demo:**

In [7]:
class Person:
    name = Typed(str)
    age = Typed(int)

    def __init__(self, name, age):
        self.name = name
        self.age = age

p = Person("Alice", 30)
print(p.name, p.age)

try:
    p.age = "thirty"
except TypeError as e:
    print("TypeError:", e)


Alice 30
TypeError: Expected int, got str


## Question 4 — `product_of_multiples` [5 marks]

Returns the product of all multiples of `factor` that are strictly less than `limit`, using a `for` loop and `range`.


In [8]:
def product_of_multiples(factor, limit):
    product = 1
    for i in range(factor, limit, factor):
        product *= i
    return product


**Demo:**

In [9]:
print(product_of_multiples(3, 15))   # 3 * 6 * 9 * 12 = 1944
print(product_of_multiples(5, 5))    # no multiples below limit -> empty product = 1
print(product_of_multiples(2, 10))   # 2 * 4 * 6 * 8 = 384


1944
1
384
